In [1]:
import pandas as pd
import numpy as np
import multiprocess
import ast
from sklearn.preprocessing import MultiLabelBinarizer
import requests
from bs4 import BeautifulSoup
import time

In [2]:
def scrape(start_page=1, n_pages=1):
    listings = pd.DataFrame(
        columns = [
            "title",
            "url",
            "price",
            "location",
            "property_type",
            "area",
            "bedrooms",
            "bathrooms",
            "status",
            "furnishing",
            "posted_on",
            "agent_name",
            "specs"
        ]
    )
    
    base_url = "https://lifenavi.com/ph/properties/search"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    end_page = start_page + n_pages
    print(f"Scraping pages {start_page} to {end_page - 1}...")

    for n in range(start_page, end_page):
        url = f"{base_url}?page={n}"
        print(f"Fetching {url}")
        
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, "html.parser")
            
            titles = soup.find_all("h2")
            
            if not titles:
                print(f"No titles found on page {n}. Stopping batch.")
                break

            for title_tag in titles:
                try:
                    # 1. Title & URL
                    link_tag = title_tag.find_parent("a")
                    if not link_tag:
                         parent = title_tag.find_parent("div")
                         if parent:
                             link_tag = parent.find_parent("a")
                    
                    listing_url = link_tag["href"] if link_tag else None
                    if listing_url and not listing_url.startswith("http"):
                        listing_url = "https://lifenavi.com" + listing_url

                    # Find Card Container
                    card = title_tag.find_parent("div")
                    if card:
                        card = card.find_parent("div")
                    if not card:
                        continue

                    # 2. Price
                    price_tag = card.find("span", class_=lambda x: x and "text-custom-red-500" in x)
                    price = price_tag.get_text(strip=True) if price_tag else np.nan

                    # 3. Location
                    location = np.nan
                    spans = card.find_all("span")
                    for s in spans:
                        txt = s.get_text(strip=True)
                        if "," in txt and "PHP" not in txt and len(txt) < 80 and "Posted" not in txt:
                             location = txt
                             break

                    # 4. Type of Property (Red Badge)
                    property_type = np.nan
                    badge = card.find("div", class_=lambda x: x and "absolute" in x)
                    if badge:
                        badge_span = badge.find("span")
                        if badge_span:
                             property_type = badge_span.get_text(strip=True)
                    
                    # 5. Extract Details
                    area = np.nan
                    bedrooms = np.nan
                    bathrooms = np.nan
                    status = np.nan
                    furnishing = np.nan
                    posted_on = np.nan
                    agent_name = np.nan

                    all_text = card.get_text("|", strip=True).split("|")
                    
                    def get_value_after_label(label, text_list):
                        for i, t in enumerate(text_list):
                            if label in t:
                                val = t.replace(label, "").strip()
                                if val:
                                    return val
                                if i + 1 < len(text_list):
                                    return text_list[i+1]
                        return np.nan

                    area = get_value_after_label("Area:", all_text)
                    bedrooms = get_value_after_label("Bedrooms:", all_text)
                    bathrooms = get_value_after_label("Bathrooms:", all_text)
                    status = get_value_after_label("Status:", all_text)
                    furnishing = get_value_after_label("Furnishing:", all_text)
                    
                    for i, t in enumerate(all_text):
                        if "Posted on" in t:
                            if i + 1 < len(all_text):
                                posted_on = all_text[i+1]

                    for i, t in enumerate(all_text):
                        if "Personal Ad" in t or "Business Ad" in t:
                            if i + 1 < len(all_text):
                                agent_name = all_text[i+1]
                    
                    if pd.isna(status):
                         avail = get_value_after_label("Available from:", all_text)
                         if avail is not np.nan:
                             status = f"Available from {avail}"

                    specs = [s.get_text(strip=True) for s in spans]

                    listing = {
                        "title": title_tag.get_text(strip=True),
                        "url": listing_url,
                        "price": price,
                        "location": location,
                        "property_type": property_type,
                        "area": area,
                        "bedrooms": bedrooms,
                        "bathrooms": bathrooms,
                        "status": status,
                        "furnishing": furnishing,
                        "posted_on": posted_on,
                        "agent_name": agent_name,
                        "specs": str(specs)
                    }
                    
                    listings.loc[len(listings)] = listing
                
                except Exception as e:
                    print(f"Error parsing a card: {e}")
                    continue
            
            time.sleep(1)

        except Exception as e:
            print(f"Failed to fetch page {n}: {e}")
    
    return listings

In [3]:
# Batch 1: Pages 1-50
print("Starting Batch 1...")
df_batch1 = scrape(start_page=1, n_pages=50)
df_batch1.to_csv("lifenavi_batch_1.csv", index=False)
print("Batch 1 saved to lifenavi_batch_1.csv")
display(df_batch1.head())

Starting Batch 1...
Scraping pages 1 to 50...
Fetching https://lifenavi.com/ph/properties/search?page=1
Fetching https://lifenavi.com/ph/properties/search?page=2
Fetching https://lifenavi.com/ph/properties/search?page=3
Fetching https://lifenavi.com/ph/properties/search?page=4
Fetching https://lifenavi.com/ph/properties/search?page=5
Fetching https://lifenavi.com/ph/properties/search?page=6
Fetching https://lifenavi.com/ph/properties/search?page=7
Fetching https://lifenavi.com/ph/properties/search?page=8
Fetching https://lifenavi.com/ph/properties/search?page=9
Fetching https://lifenavi.com/ph/properties/search?page=10
Fetching https://lifenavi.com/ph/properties/search?page=11
Fetching https://lifenavi.com/ph/properties/search?page=12
Fetching https://lifenavi.com/ph/properties/search?page=13
Fetching https://lifenavi.com/ph/properties/search?page=14
Fetching https://lifenavi.com/ph/properties/search?page=15
Fetching https://lifenavi.com/ph/properties/search?page=16
Fetching https://li

,title,url,price,location,property_type,area,bedrooms,bathrooms,status,furnishing,posted_on,agent_name,specs
0,"Condo For Rent in Timog Avenue, Quezon City",https://lifenavi.com/ph/properties/quezon-city...,"PHP 16,000/ month","Timog Avenue, Diliman, Quezon City, Metro Manila",NaN,21 m²,Studio,1,Available from 21 Dec,NaN,21 Dec 22:21,Armien C.,"['PHP 16,000/ month', '/ month', 'Timog Avenue..."
1,"Condo For Sale in Lapu-Lapu, Central Visayas",https://lifenavi.com/ph/properties/lapu-lapu/f...,"PHP 5,685,000","Lapu-Lapu, Central Visayas",NaN,28 m²,Studio,1,Available from 21 Dec,NaN,21 Dec 20:46,Nelson B.,"['PHP 5,685,000', 'Lapu-Lapu, Central Visayas'..."
2,"House and Lot For Sale in San Jose del Monte, ...",https://lifenavi.com/ph/properties/san-jose-de...,"PHP 1,500,000,000","San Jose del Monte, Central Luzon",NaN,112 m²,1,2,Available from 22 Dec,NaN,21 Dec 02:05,Christian G.,"['PHP 1,500,000,000', 'San Jose del Monte, Cen..."
3,"Condo For Rent in 175, Shaw Boulevard, Pasig",https://lifenavi.com/ph/properties/pasig/for-r...,"PHP 23,000/ month","175, Shaw Boulevard, Pasig, Metro Manila",NaN,24 m²,1,1,Available from 19 Dec,NaN,19 Dec 15:41,Armien C.,"['PHP 23,000/ month', '/ month', '175, Shaw Bo..."
4,"Townhouse For Sale in Mabalacat City, Central ...",https://lifenavi.com/ph/properties/mabalacat-c...,"PHP 3,300,000","Mabalacat City, Central Luzon",NaN,54 m²,2,1,Available from 18 Dec,NaN,18 Dec 23:32,Maria soleil C.,"['PHP 3,300,000', 'Mabalacat City, Central Luz..."


In [4]:
# Batch 2: Pages 51-100
print("Starting Batch 2...")
df_batch2 = scrape(start_page=51, n_pages=50)
df_batch2.to_csv("lifenavi_batch_2.csv", index=False)
print("Batch 2 saved to lifenavi_batch_2.csv")
display(df_batch2.head())

Starting Batch 2...
Scraping pages 51 to 100...
Fetching https://lifenavi.com/ph/properties/search?page=51
Fetching https://lifenavi.com/ph/properties/search?page=52
Fetching https://lifenavi.com/ph/properties/search?page=53
Fetching https://lifenavi.com/ph/properties/search?page=54
Fetching https://lifenavi.com/ph/properties/search?page=55
Fetching https://lifenavi.com/ph/properties/search?page=56
Fetching https://lifenavi.com/ph/properties/search?page=57
Fetching https://lifenavi.com/ph/properties/search?page=58
Fetching https://lifenavi.com/ph/properties/search?page=59
Fetching https://lifenavi.com/ph/properties/search?page=60
Fetching https://lifenavi.com/ph/properties/search?page=61
Fetching https://lifenavi.com/ph/properties/search?page=62
Fetching https://lifenavi.com/ph/properties/search?page=63
Fetching https://lifenavi.com/ph/properties/search?page=64
Fetching https://lifenavi.com/ph/properties/search?page=65
Fetching https://lifenavi.com/ph/properties/search?page=66
Fetching

,title,url,price,location,property_type,area,bedrooms,bathrooms,status,furnishing,posted_on,agent_name,specs
0,"House and Lot For Sale in Greenwoods Avenue, C...",https://lifenavi.com/ph/properties/cainta/for-...,"PHP 11,500,000","Greenwoods Avenue, Cainta, Metro Manila",NaN,160 m²,4,2,Available from 28 Nov,NaN,01 Dec 17:00,Housing I.,"['PHP 11,500,000', 'Greenwoods Avenue, Cainta,..."
1,"Condo For Sale in East Raya Gardens, Mercedes ...",https://lifenavi.com/ph/properties/pasig/for-s...,"PHP 4,500,000","East Raya Gardens, Mercedes Avenue, Pasig, Met...",NaN,64 m²,2,2,Available from 9 Oct,NaN,01 Dec 17:00,Housing I.,"['PHP 4,500,000', 'East Raya Gardens, Mercedes..."
2,"Condo For Rent in Taguig, Metro Manila",https://lifenavi.com/ph/properties/taguig/for-...,"PHP 19,000/ month","Taguig, Metro Manila",NaN,27.33 m²,1,1,Available from 12 May,NaN,01 Dec 17:00,Housing I.,"['PHP 19,000/ month', '/ month', 'Taguig, Metr..."
3,"Condo For Sale in General Romulo Avenue, Quezo...",https://lifenavi.com/ph/properties/quezon-city...,"PHP 3,500,000","General Romulo Avenue, Cubao, Quezon City, Met...",NaN,28 m²,Studio,1,Available from 11 Aug,NaN,01 Dec 16:59,Housing I.,"['PHP 3,500,000', 'General Romulo Avenue, Cuba..."
4,"House and Lot For Sale in 17, Green Meadows Av...",https://lifenavi.com/ph/properties/quezon-city...,"PHP 60,000,000","17, Green Meadows Avenue, Quezon City, Metro M...",NaN,400 m²,4,4,Available from 28 Mar,NaN,01 Dec 16:59,Housing I.,"['PHP 60,000,000', '17, Green Meadows Avenue, ..."


In [5]:
# Batch 3: Pages 101-150
print("Starting Batch 3...")
df_batch3 = scrape(start_page=101, n_pages=50)
df_batch3.to_csv("lifenavi_batch_3.csv", index=False)
print("Batch 3 saved to lifenavi_batch_3.csv")
display(df_batch3.head())

Starting Batch 3...
Scraping pages 101 to 150...
Fetching https://lifenavi.com/ph/properties/search?page=101
Fetching https://lifenavi.com/ph/properties/search?page=102
Fetching https://lifenavi.com/ph/properties/search?page=103
Fetching https://lifenavi.com/ph/properties/search?page=104
Fetching https://lifenavi.com/ph/properties/search?page=105
Fetching https://lifenavi.com/ph/properties/search?page=106
Fetching https://lifenavi.com/ph/properties/search?page=107
Fetching https://lifenavi.com/ph/properties/search?page=108
Fetching https://lifenavi.com/ph/properties/search?page=109
Fetching https://lifenavi.com/ph/properties/search?page=110
Fetching https://lifenavi.com/ph/properties/search?page=111
Fetching https://lifenavi.com/ph/properties/search?page=112
Fetching https://lifenavi.com/ph/properties/search?page=113
Fetching https://lifenavi.com/ph/properties/search?page=114
Fetching https://lifenavi.com/ph/properties/search?page=115
Fetching https://lifenavi.com/ph/properties/search?

,title,url,price,location,property_type,area,bedrooms,bathrooms,status,furnishing,posted_on,agent_name,specs
0,"Condo For Rent in corner, 28th Street, Taguig",https://lifenavi.com/ph/properties/taguig/for-...,"PHP 170,000/ month","corner, 28th Street, Taguig, Metro Manila",NaN,98 m²,2,2,Available from 19 Dec,NaN,17 Nov 13:16,Housing I.,"['PHP 170,000/ month', '/ month', 'corner, 28t..."
1,"Condo For Rent in 4th Avenue, Makati City",https://lifenavi.com/ph/properties/makati-city...,"PHP 300,000/ month","4th Avenue, Makati City, Metro Manila",NaN,298 m²,3,2,Available from 19 Dec,NaN,17 Nov 13:16,Housing I.,"['PHP 300,000/ month', '/ month', '4th Avenue,..."
2,"Condo For Rent in Rockwell Drive, Makati City",https://lifenavi.com/ph/properties/makati-city...,"PHP 120,000/ month","Rockwell Drive, Makati City, Metro Manila",NaN,98 m²,2,2,Available from 19 Dec,NaN,17 Nov 13:16,Housing I.,"['PHP 120,000/ month', '/ month', 'Rockwell Dr..."
3,"Condo For Rent in 98, Perea, Makati City",https://lifenavi.com/ph/properties/makati-city...,"PHP 165,000/ month","98, Perea, Makati City, Metro Manila",NaN,130 m²,2,2,Available from 19 Dec,NaN,17 Nov 13:16,Housing I.,"['PHP 165,000/ month', '/ month', '98, Perea, ..."
4,"Condo For Rent in Cor, 11th Avenue, Taguig",https://lifenavi.com/ph/properties/taguig/for-...,"PHP 130,000/ month","Cor, 11th Avenue, Taguig, Metro Manila",NaN,82 m²,2,2,Available from 19 Dec,NaN,17 Nov 13:16,Housing I.,"['PHP 130,000/ month', '/ month', 'Cor, 11th A..."


In [6]:
# Batch 4: Pages 151-200 (covers the rest of ~174 pages)
print("Starting Batch 4...")
df_batch4 = scrape(start_page=151, n_pages=50)
df_batch4.to_csv("lifenavi_batch_4.csv", index=False)
print("Batch 4 saved to lifenavi_batch_4.csv")
display(df_batch4.head())

Starting Batch 4...
Scraping pages 151 to 200...
Fetching https://lifenavi.com/ph/properties/search?page=151
Fetching https://lifenavi.com/ph/properties/search?page=152
Fetching https://lifenavi.com/ph/properties/search?page=153
Fetching https://lifenavi.com/ph/properties/search?page=154
Fetching https://lifenavi.com/ph/properties/search?page=155
Fetching https://lifenavi.com/ph/properties/search?page=156
Fetching https://lifenavi.com/ph/properties/search?page=157
Fetching https://lifenavi.com/ph/properties/search?page=158
Fetching https://lifenavi.com/ph/properties/search?page=159
Fetching https://lifenavi.com/ph/properties/search?page=160
Fetching https://lifenavi.com/ph/properties/search?page=161
Fetching https://lifenavi.com/ph/properties/search?page=162
Fetching https://lifenavi.com/ph/properties/search?page=163
Fetching https://lifenavi.com/ph/properties/search?page=164
Fetching https://lifenavi.com/ph/properties/search?page=165
Fetching https://lifenavi.com/ph/properties/search?

,title,url,price,location,property_type,area,bedrooms,bathrooms,status,furnishing,posted_on,agent_name,specs
0,"Warehouse For Sale in City of Santa Rosa, Cala...",https://lifenavi.com/ph/properties/city-of-san...,"PHP 865,800,000","City of Santa Rosa, Calabarzon",NaN,5000 m²,NaN,NaN,Vacant,Semi-Furnished,04 Nov 13:54,Housing I.,"['PHP 865,800,000', 'City of Santa Rosa, Calab..."
1,"For Sale in Makati City, Metro Manila",https://lifenavi.com/ph/properties/makati-city...,"PHP 3,000,000,000","Makati City, Metro Manila",NaN,10000 m²,NaN,NaN,Vacant,Semi-Furnished,04 Nov 13:54,Housing I.,"['PHP 3,000,000,000', 'Makati City, Metro Mani..."
2,"For Sale in Rockwell Drive, Makati City",https://lifenavi.com/ph/properties/makati-city...,"PHP 280,000,000","Rockwell Drive, Makati City, Metro Manila",NaN,1720 m²,NaN,NaN,Vacant,Semi-Furnished,04 Nov 13:54,Housing I.,"['PHP 280,000,000', 'Rockwell Drive, Makati Ci..."
3,"Condo For Rent in cor, 7th Avenue, Taguig",https://lifenavi.com/ph/properties/taguig/for-...,"PHP 45,000/ month","cor, 7th Avenue, Taguig, Metro Manila",NaN,39 m²,1,1,Available from 4 Aug,NaN,04 Nov 13:54,Housing I.,"['PHP 45,000/ month', '/ month', 'cor, 7th Ave..."
4,"Condo For Sale in corner, Sen. Gil J. Puyat Av...",https://lifenavi.com/ph/properties/pasay-city/...,"PHP 26,740,000","corner, Sen. Gil J. Puyat Avenue, Pasay City, ...",NaN,84 m²,2,1,Available from 5 Aug,NaN,04 Nov 13:54,Housing I.,"['PHP 26,740,000', 'corner, Sen. Gil J. Puyat ..."


In [7]:
# Optional: Merge all batches
# df_all = pd.concat([df_batch1, df_batch2, df_batch3, df_batch4], ignore_index=True)
# df_all.to_csv("lifenavi_all_properties.csv", index=False)
# print(f"Total merged listings: {len(df_all)}")